In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

In [3]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

In [4]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]=LANGSMITH_PROJECT

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [7]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [8]:
while True:
    question=input("type your question. if you want to quit the chat write quit")
    if question !="quit":
        print(llm.invoke(question).content)
    else:
        print("goodbye take care yourself")
        break

Hello Pragi, it's nice to meet you. Is there something I can help you with or would you like to chat?
I don't know your name. I'm a large language model, I don't have the ability to recall personal information about individuals, and our conversation just started, so I haven't received any information about you. If you'd like to share your name, I'd be happy to chat with you!
goodbye take care yourself


In [9]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [10]:
store={}

In [11]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [12]:
config = {"configurable": {"session_id": "firstchat"}}

In [13]:
model_with_memory=RunnableWithMessageHistory(llm,get_session_history)

d:\AI-ML\vscode\projects_dir\LangGraph_Shakedown\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
model_with_memory.invoke(("Hi! I'm prakadeesh"),config=config).content

"Hello Prakadeesh, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [15]:
model_with_memory.invoke(("tell me what is my name?"),config=config).content

'Your name is Prakadeesh.'

In [16]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough , RunnableLambda
from langchain_core.output_parsers import StrOutputParser

### Reading the txt files from source directory

loader = DirectoryLoader('data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

### Creating Chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

###  BGE Embddings

'''from langchain.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
'''

### Creating Retriever using Vector DB

db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

In [17]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [18]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

In [19]:
question ="what is llama3? can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

Based on the provided context, Llama 3 appears to be a product or technology developed by Meta. Here are three important points about Llama 3:

1. **Robust Safety Ecosystem**: Llama 3 utilizes a robust safety ecosystem, suggesting that it has been designed with safety and security in mind.
2. **Meta's Product**: Llama 3 is Meta's product, indicating that it is a technology or tool developed by the company.
3. **Advanced Architecture and Training**: Llama 3 was built using advanced architecture and training methods, and is capable of being deployed on enterprise-grade server clusters, implying that it is a sophisticated and powerful technology.


# Let's Start with Tools and Agents

In [20]:
import wikipedia

wikipedia.set_user_agent(
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/125.0.0.0 Safari/537.36 "
    "LangGraphShakedown/1.0"
)

In [21]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [22]:
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=100)

In [23]:
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [24]:
tool.name

'wikipedia'

In [25]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [26]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [27]:
tool.return_direct

False

In [28]:
print(tool.run({"query": "langchain"}))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [29]:
from pydantic import BaseModel, Field

In [30]:
class WikiInputs(BaseModel):
    query: str = Field(description="query to look up in Wikipedia, should be 3 or less words")

In [31]:
tool=WikipediaQueryRun(
    name="wiki-tool",
    description="look up things in wikipedia",
    args_schema=WikiInputs,
    api_wrapper=api_wrapper, 
    return_direct=True,
)

In [32]:
tool.name

'wiki-tool'

In [33]:
tool.description

'look up things in wikipedia'

In [34]:
tool.args

{'query': {'description': 'query to look up in Wikipedia, should be 3 or less words',
  'title': 'Query',
  'type': 'string'}}

In [35]:
tool.return_direct

True

In [36]:
print(tool.run("langchain"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# youtube search tool

In [37]:
from langchain_community.tools import YouTubeSearchTool

In [38]:
tool = YouTubeSearchTool()

In [39]:
tool.name

'youtube_search'

In [40]:
tool.description

'search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional'

In [41]:
tool.run("sunny savita")

"['https://www.youtube.com/watch?v=Tf2ZzrCBJUI&pp=ygUMc3Vubnkgc2F2aXRh', 'https://www.youtube.com/watch?v=ENzZuvahKwc&pp=ygUMc3Vubnkgc2F2aXRh']"

In [42]:
from langchain_community.tools.tavily_search import TavilySearchResults

tool = TavilySearchResults()

C:\Users\pragi\AppData\Local\Temp\ipykernel_7132\651679412.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults()


In [43]:
tool.invoke({"query": "What happened in the latest burning man floods"})

[{'title': 'Burning Man flooding: What happened to stranded festivalgoers? - ABC News',
  'url': 'https://abcnews.com/US/burning-man-flooding-happened-stranded-festivalgoers/story?id=102908331',
  'content': '## Live\n\n## Video\n\n## Shows\n\n## Good Morning America\n\n## ShopGMA\n\n## Stream on\n\nstream logo\n\n# Burning Man flooding: What happened to stranded festivalgoers?\n\nSome 64,000 people were still on site Monday as the exodus began.\n\nTens of thousands of Burning Man attendees are now able to leave the festival after a downpour and massive flooding left them stranded over the weekend.\n\nThe festival, held in the Black Rock Desert in northwestern Nevada, was attended by more than 70,000 people last year, and just as many were expected this year. Burning Man began on Aug. 28 and was scheduled to run through Sept. 4.\n\nOne person died at the festival amid the unusual weather conditions, the Pershing County Sheriff\'s Office confirmed Sunday morning in a statement. The deat

In [44]:
from langchain_classic import hub

In [45]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

In [46]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [47]:
tavily_tool = TavilySearchResults()

In [48]:
tools = [tavily_tool]

In [49]:
agent = create_tool_calling_agent(llm, tools, prompt)

In [50]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [51]:
print(agent_executor.invoke({"input": "who was mahatma gandhi"}))



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'Mahatma Gandhi biography'}`


[{'title': 'Mahatma Gandhi | Britannica - Britannica', 'url': 'https://www.britannica.com/biography/Mahatma-Gandhi', 'content': "Mahatma Gandhi (born October 2, 1869, Porbandar, India—died January 30, 1948, Delhi) was an Indian lawyer, politician, social activist, and writer who became the leader of the Indian Independence Movement against British rule. As such, he came to be considered “the father of the nation.” Gandhi is internationally esteemed for his doctrine of nonviolent protest (satyagraha) to achieve political and social progress. [...] SUBSCRIBE\n\nAsk the Chatbot Games & Quizzes History & Society Science & Tech Biographies Animals & Nature Geography & Travel Arts & Culture ProCon Money Videos\n\nBritannica AI Icon\n\nMahatma Gandhi\n\nMahatma Gandhi\nMohandas Karamchand Gandhi, popularly known as the Mahatma (“Great Soul”), in London, 1931.\n\n# Mah

# Create our custom agent and custom tools

In [52]:
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool, StructuredTool, tool

In [53]:
@tool
def search(query: str) -> str:
    """Look up things online."""
    return "LangChain"

In [54]:
print(search.name)
print(search.description)
print(search.args)

search
Look up things online.
{'query': {'title': 'Query', 'type': 'string'}}


In [55]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [56]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [57]:
class SearchInput(BaseModel):
    query: str = Field(description="should be a search query")

In [58]:
@tool("search-tool", args_schema=SearchInput, return_direct=True)
def search(query: str) -> str:
    """Look up things online."""
    return "LangChain"

In [59]:
print(search.name)
print(search.description)
print(search.args)
print(search.return_direct)

search-tool
Look up things online.
{'query': {'description': 'should be a search query', 'title': 'Query', 'type': 'string'}}
True


In [60]:
from typing import Optional, Type

from langchain_core.callbacks import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)

In [61]:
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

In [62]:
get_word_length.invoke("abc")

3

In [63]:
tools = [tavily_tool, get_word_length]

In [64]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but don't know current events",
        ),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [65]:
llm_with_tools = llm.bind_tools(tools)

In [66]:
from langchain_classic.agents.format_scratchpad.openai_tools import format_to_openai_tool_messages
from langchain_classic.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

In [67]:
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

In [68]:
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [69]:
 list(agent_executor.stream({"input": "How many letters in the word eudca"}))



> Entering new None chain...

Invoking: `get_word_length` with `{'word': 'eudca'}`


5The word "eudca" has 5 letters.

> Finished chain.


[{'actions': [ToolAgentAction(tool='get_word_length', tool_input={'word': 'eudca'}, log="\nInvoking: `get_word_length` with `{'word': 'eudca'}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': '6y06mp7h9', 'function': {'arguments': '{"word":"eudca"}', 'name': 'get_word_length'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq', 'finish_reason': 'tool_calls', 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand'}, id='lc_run--019ef580-774e-7b50-8967-e637c6f28bc6', tool_calls=[{'name': 'get_word_length', 'args': {'word': 'eudca'}, 'id': '6y06mp7h9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 352, 'output_tokens': 17, 'total_tokens': 369}, tool_call_chunks=[{'name': 'get_word_length', 'args': '{"word":"eudca"}', 'id': '6y06mp7h9', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')], tool_call_id='6y06mp7h9')],
  'messag

In [70]:
llm.invoke("How many letters in the word educa")

AIMessage(content='The word "educa" has 5 letters: E-D-U-C-A.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 43, 'total_tokens': 61, 'completion_time': 0.046317343, 'completion_tokens_details': None, 'prompt_time': 0.005697614, 'prompt_tokens_details': None, 'queue_time': 0.499891609, 'total_time': 0.052014957}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef580-78c6-7681-91fd-5c934c96840e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 18, 'total_tokens': 61})

In [71]:
from langchain_core.prompts import MessagesPlaceholder

MEMORY_KEY = "chat_history"
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but bad at calculating lengths of words.",
        ),
        MessagesPlaceholder(variable_name=MEMORY_KEY),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [72]:
chat_history = []

In [73]:
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
) 

In [74]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [75]:
from langchain_core.messages import AIMessage, HumanMessage

input1 = "how many letters in the word educa?"
result = agent_executor.invoke({"input": input1, "chat_history": chat_history})



> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'educa'}`


5The word "educa" has 5 letters: e-d-u-c-a.

> Finished chain.


In [76]:
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)

agent_executor.invoke({"input": "is that a real word?", "chat_history": chat_history})



> Entering new AgentExecutor chain...

Invoking: `get_word_length` with `{'word': 'educa'}`


5The word "educa" is not a commonly used word in English. However, it is a prefix or a root word that is related to education. A more common word would be "educate" or "education". 

If you meant to ask about the word "educate" or "education", I can help you with that. The word "educate" has 7 letters: e-d-u-c-a-t-e. The word "education" has 9 letters: e-d-u-c-a-t-i-o-n.

> Finished chain.


{'input': 'is that a real word?',
 'chat_history': [HumanMessage(content='how many letters in the word educa?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The word "educa" has 5 letters: e-d-u-c-a.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'The word "educa" is not a commonly used word in English. However, it is a prefix or a root word that is related to education. A more common word would be "educate" or "education". \n\nIf you meant to ask about the word "educate" or "education", I can help you with that. The word "educate" has 7 letters: e-d-u-c-a-t-e. The word "education" has 9 letters: e-d-u-c-a-t-i-o-n.'}